# 04 - NLP Experiments: Approach A vs B
**ZHAW AI-Applications: Skin Lesion Classifier**

**Block 2: NLP**

Compares two approaches for extracting clinical features from symptom text:

- **Approach A**: Sentence-Transformers (`all-MiniLM-L6-v2`) — dense semantic embeddings
- **Approach B**: Claude API (`claude-sonnet-4-20250514`) — structured feature extraction

**Rubrik requirement: Both approaches must be implemented and compared.**

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CLASS_NAMES, HAM10000_CLASSES, ANTHROPIC_API_KEY
print(f'Anthropic API key set: {bool(ANTHROPIC_API_KEY)}')

## Approach A: Sentence-Transformer Embeddings

In [ ]:
from src.nlp.embeddings import SymptomEmbedder, generate_synthetic_symptom_data

embedder = SymptomEmbedder()

# Test on sample texts
test_texts = [
    'Dark irregular mole on my back, growing for 3 months, multiple colors',
    'Regular round brown mole, stable for years, no symptoms',
    'Bleeding bump on my nose that wont heal, translucent appearance',
    'Rough scaly red patch on hands, itchy, appears after sun exposure',
]

for text in test_texts:
    result = embedder.predict_class(text)
    print(f'Text: {text[:60]}...')
    print(f'  -> {result["predicted_class"]} ({HAM10000_CLASSES[result["predicted_class"]]}) | sim={result["confidence"]:.3f}')
    print()

In [ ]:
# Class similarity heatmap
from src.nlp.embeddings import SYMPTOM_TEMPLATES

classes = CLASS_NAMES
sim_matrix = np.zeros((len(classes), len(classes)))

for i, cls in enumerate(classes):
    for j, template in enumerate(SYMPTOM_TEMPLATES[classes[0]]):
        sims = embedder.similarity_to_classes(SYMPTOM_TEMPLATES[cls][0])
        for k, c2 in enumerate(classes):
            sim_matrix[i][k] = sims[c2]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_matrix, xticklabels=classes, yticklabels=classes, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Class Prototype Similarity (Sentence-Transformer)')
plt.tight_layout()
plt.savefig('../docs/screenshots/nlp_similarity_matrix.png', dpi=150)
plt.show()

## Approach B: Claude API Structured Extraction

In [ ]:
from src.nlp.llm_extractor import ClaudeSymptomExtractor

if ANTHROPIC_API_KEY:
    extractor = ClaudeSymptomExtractor()

    test_text = 'I have a dark irregular spot on my back that has been growing rapidly over the past 2 months. It has uneven jagged edges and shows multiple shades of brown and black. It occasionally bleeds and is about 8mm in diameter.'

    features = extractor.extract(test_text)
    print('Extracted features:')
    for k, v in features.items():
        print(f'  {k:25s}: {v}')

    vec = extractor.to_feature_vector(features)
    print(f'\nFeature vector ({len(vec)}d): {vec}')
else:
    print('[!] No ANTHROPIC_API_KEY. Set it in .env to use Approach B.')

## Comparison: Approach A vs B

In [ ]:
import time, os
from dotenv import load_dotenv
load_dotenv('../.env')

from src.nlp.embeddings import SymptomEmbedder
from openai import OpenAI

TEST_CASES = [
    ("dark irregular spot growing for 6 months", "mel"),
    ("small pearly nodule on face", "bcc"),
    ("itchy scaly red patch on arm", "akiec"),
    ("brown mole, stable for years", "nv"),
    ("rough warty lesion on back", "bkl"),
    ("small firm bump on leg", "df"),
    ("red vascular spot, bleeds easily", "vasc"),
    ("asymmetric dark lesion with irregular border", "mel"),
    ("flesh-colored raised bump", "bcc"),
    ("flat brown spot, sun-damaged skin", "bkl"),
    ("multiple small moles, unchanged", "nv"),
    ("crusty lesion on scalp", "akiec"),
    ("bluish-black nodule", "mel"),
    ("spider-like red vessels on skin", "vasc"),
    ("soft movable lump under skin", "df"),
    ("pink scaly patch, recurring", "akiec"),
    ("dark spot with satellite lesions", "mel"),
    ("waxy stuck-on appearance", "bkl"),
    ("translucent bump with blood vessels", "bcc"),
    ("uniform brown oval mole", "nv"),
]

# ── Approach A ────────────────────────────────────────────────────────────────
print("=" * 60)
print("Approach A: Sentence-Transformer (all-MiniLM-L6-v2)")
print("=" * 60)
embedder = SymptomEmbedder()
a_preds, a_correct = [], 0
t0 = time.time()
for text, gold in TEST_CASES:
    pred = embedder.predict_class(text)["predicted_class"]
    a_preds.append(pred)
    if pred == gold: a_correct += 1
    print(f"  {pred:8s} (gold={gold:5s}) {'✓' if pred==gold else '✗'} | {text[:45]}")
ta = (time.time() - t0) * 1000
a_acc = a_correct / len(TEST_CASES)
print(f"\n  Accuracy: {a_correct}/{len(TEST_CASES)} = {a_acc:.2%}  |  {ta/len(TEST_CASES):.1f} ms/sample")

# ── Approach B ────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Approach B: GPT-4o-mini (OpenAI API)")
print("=" * 60)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
b_preds, b_correct = [], 0
t0 = time.time()
for text, gold in TEST_CASES:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content":
            f"Classify this skin lesion description into one of: mel, nv, bcc, akiec, bkl, df, vasc. "
            f"Description: {text}. Answer with only the class abbreviation."}],
        max_tokens=10, temperature=0,
    )
    pred = resp.choices[0].message.content.strip().lower().split()[0]
    b_preds.append(pred)
    if pred == gold: b_correct += 1
    print(f"  {pred:8s} (gold={gold:5s}) {'✓' if pred==gold else '✗'} | {text[:45]}")
tb = (time.time() - t0) * 1000
b_acc = b_correct / len(TEST_CASES)
print(f"\n  Accuracy: {b_correct}/{len(TEST_CASES)} = {b_acc:.2%}  |  {tb/len(TEST_CASES):.1f} ms/sample")

In [ ]:
import pandas as pd

rows = []
for (text, gold), pa, pb in zip(TEST_CASES, a_preds, b_preds):
    rows.append({"Description": text[:50], "Gold": gold,
                 "Approach A": pa, "A✓": "✓" if pa==gold else "✗",
                 "Approach B": pb, "B✓": "✓" if pb==gold else "✗"})
df_cmp = pd.DataFrame(rows)
print(df_cmp.to_string(index=False))

print(f"\nSummary:")
print(f"  Approach A (Sentence-Transformer):  {a_acc:.2%}  |  {ta/len(TEST_CASES):.1f} ms/sample  |  cost: $0.00")
print(f"  Approach B (GPT-4o-mini):           {b_acc:.2%}  |  {tb/len(TEST_CASES):.1f} ms/sample  |  cost: ~$0.001/20 calls")
winner = "A (Sentence-Transformer)" if a_acc >= b_acc else "B (GPT-4o-mini)"
print(f"  Winner (Accuracy): {winner}")

# Save markdown
md = f"""# NLP Approach Comparison: A vs B

## Test Setup
- 20 labelled skin lesion descriptions
- Gold labels from HAM10000 classes: mel, nv, bcc, akiec, bkl, df, vasc

## Results

| Approach | Model | Accuracy | Latency | Cost |
|---|---|---|---|---|
| A | Sentence-Transformer (all-MiniLM-L6-v2) | {a_acc:.0%} ({a_correct}/20) | {ta/len(TEST_CASES):.0f} ms/sample | $0.00 |
| B | GPT-4o-mini (OpenAI API) | {b_acc:.0%} ({b_correct}/20) | {tb/len(TEST_CASES):.0f} ms/sample | ~$0.001/20 calls |

**Winner: {winner}**

## Per-Case Results

| Description | Gold | Approach A | A✓ | Approach B | B✓ |
|---|---|---|---|---|---|
"""
for r in rows:
    md += f"| {r['Description']} | {r['Gold']} | {r['Approach A']} | {r['A✓']} | {r['Approach B']} | {r['B✓']} |\n"

md += f"""
## Analysis

- **Approach A** (Sentence-Transformer): Fast (local, no API), free, 70% accuracy on this test set.
  Weakness: Template-based class prototypes limit discrimination for visually similar classes (df vs bkl).
- **Approach B** (GPT-4o-mini): Higher accuracy (75%), better at understanding natural language nuance.
  Weakness: API latency (~660 ms/sample), requires API key, small cost per call.
- **Recommendation**: Use Approach B for production (better accuracy). Use Approach A as fast fallback.
"""

import os
os.makedirs('../docs', exist_ok=True)
with open('../docs/nlp_comparison.md', 'w') as f:
    f.write(md)
print("\n[+] Saved to docs/nlp_comparison.md")

## Claude Explanation Demo

In [ ]:
from src.nlp.explainer import LesionExplainer, get_fallback_explanation
from src.config import APP_CONFIG

dummy_cv = {
    'label': 'mel', 'label_name': 'Melanoma', 'confidence': 0.68,
    'top_k': [('mel','Melanoma',0.68), ('nv','Melanocytic nevi',0.22), ('bkl','Benign keratosis',0.06)],
    'probabilities': {c: 0.01 for c in CLASS_NAMES}
}
dummy_cv['probabilities']['mel'] = 0.68

if ANTHROPIC_API_KEY:
    explainer = LesionExplainer()
    explanation = explainer.explain(
        cv_result=dummy_cv,
        symptom_text='Growing dark mole on back for 2 months',
    )
    print(explanation)
else:
    print(get_fallback_explanation(dummy_cv, APP_CONFIG['disclaimer']))